In [1]:
import pandas as pd
from pathlib import Path
import os
from datetime import datetime,timedelta
today = datetime.today().date()
yesterday = datetime.now() - timedelta(days=2)


In [2]:
def create_error_file(dataframe: pd.DataFrame, filename: str):
    if dataframe.shape[0]>0:
        output_dir = Path("output")
        output_dir.mkdir(exist_ok=True)
        file_path = output_dir / f"[{today} - Anomaly] {filename}.csv"

        dataframe.to_csv(file_path, index=False)
    
    
    

In [3]:
def extract_compoundname(package):
    
    if "-" in package:
        return package.split("-",1)[1]
    elif "El Gouna" in package:
        return "El Gouna"
    else:
        return None


In [4]:
def extract_provider(package):
    package = str(package).lower()

    bein_keywords = [
        "bein",
        "afcon",
        "euro",
        
    ]

    if any(keyword in package for keyword in bein_keywords):
        return "beIN"

    if "osn" in package:
        return "OSN"
    
    if "fta" in package:
        return "FTA"

    return "N/A"

In [5]:
def load_files(path,date):

    folder = Path(path)
    files = [f for f in folder.iterdir() if f.is_file()]
    columns = ["SUBSCRIBE_SERVICE_ID",	"SERVICE_MENU_ID",	"SUBSCRIPTION_DATE",	"EXPIRE_DATE",	"WORK_PHONE",	"CUSTOMER_ID",	"NAME",	"DEVICE_ID",	"MAC_ADDRESS",	"SOURCEFILE"]
    all = pd.DataFrame(columns=columns)
    for index, file in enumerate(files, start=1):
        
        data = pd.read_csv(os.path.join(folder,file.name),dtype='str')
        
        data['SOURCEFILE'] = file.name
        data['COMPOUND'] = data['NAME'].apply(extract_compoundname)
        compound_from_filename  = (
            file.name
            .replace("CNE_", "")
            .replace("Minerva.csv", "")
            .replace("_", "")
        )

        #data.loc[data['COMPOUND'].isna(),'COMPOUND'] = compound_from_filename
        # data = data.drop(columns=['compound','COMPOUND','SOURCEFILE'],errors="ignore")
        # data.loc[data['EXPIRE_DATE'].isna(),'EXPIRE_DATE'] = '1/1/2070'
        # data.loc[data['EXPIRE_DATE']=='1/1/2070','EXPIRE_DATE'] = None
        # data.to_csv(os.path.join(folder,file.name),index=False)
        all = pd.concat([all,data],ignore_index=False)
        all['PROVIDER'] = all['NAME'].apply(extract_provider)
        all['EXPIRE_DATE'] = pd.to_datetime(all["EXPIRE_DATE"],dayfirst = True,errors='coerce')
        all.to_csv(f'Orange All {date}.csv')
    return all






Check for missing MAC ADDRESS

In [6]:
all_old = load_files(r"C:\Users\mturky\Documents\orange data\1-6-2026", '1-6-2026')
all_new = load_files(r"C:\Users\mturky\Documents\orange data\8-6-2026",'8-6-2026')

bein_old = all_old.loc[all_old['NAME'].str.lower().str.contains('bein')]
bein_new = all_new.loc[all_new['NAME'].str.lower().str.contains('bein')]

missing = bein_old.loc[~bein_old['MAC_ADDRESS'].isin(bein_new['MAC_ADDRESS'])]
missing_future_date = missing.loc[missing['EXPIRE_DATE']>'2026-06-01']
missing_empty_date = missing.loc[missing['EXPIRE_DATE'].isna()]

create_error_file (missing_future_date,"Missing with future expire date")
create_error_file(missing_empty_date,"Missing with empty expire date")

In [7]:
bein_new.columns

Index(['SUBSCRIBE_SERVICE_ID', 'SERVICE_MENU_ID', 'SUBSCRIPTION_DATE',
       'EXPIRE_DATE', 'WORK_PHONE', 'CUSTOMER_ID', 'NAME', 'DEVICE_ID',
       'MAC_ADDRESS', 'SOURCEFILE', 'COMPOUND', 'PROVIDER'],
      dtype='object')

Check for missing DUPLICATES

In [8]:
agg = bein_new.groupby(['WORK_PHONE','MAC_ADDRESS','SOURCEFILE']).agg(count = ('MAC_ADDRESS','count')).reset_index()
agg = agg.loc[agg['count']>1]
agg = agg.astype(str)

create_error_file(agg[['WORK_PHONE','MAC_ADDRESS','SOURCEFILE']], 'Audit sample')

Empty MAC_ADDRESS

In [9]:
empty_mac = bein_new.loc[bein_new['MAC_ADDRESS'].isna()]
create_error_file(empty_mac,"Empty MAC ADDRESS")

Empty EXPIRE DATE

In [10]:
empty_expire = bein_new.loc[bein_new['EXPIRE_DATE'].isna()]
create_error_file(empty_expire,"Empty EXPIRE DATE")

Expired Files

In [11]:
empty_expire = bein_new.loc[bein_new['EXPIRE_DATE']< yesterday ]
create_error_file(empty_expire,"Expired contracts")

In [12]:
all_new.columns

Index(['SUBSCRIBE_SERVICE_ID', 'SERVICE_MENU_ID', 'SUBSCRIPTION_DATE',
       'EXPIRE_DATE', 'WORK_PHONE', 'CUSTOMER_ID', 'NAME', 'DEVICE_ID',
       'MAC_ADDRESS', 'SOURCEFILE', 'COMPOUND', 'PROVIDER'],
      dtype='object')